In [ ]:
from utils import *
from plots import *
import pickle, glob
import scipy.io as sio
from scipy.stats import spearmanr
from joblib import Parallel, delayed
import nibabel as nib
from tqdm import tqdm
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


In [ ]:
mmpi = pd.read_csv('./mmpi/Iowa_MMPI2RF_byNandi.csv').set_index('ID')
mmpi

In [ ]:
# Unified MMPI label source from utils.py
# Switch label style globally by editing utils.MMPI_LABEL_STYLE
scale_keys = MMPI_SCALE_KEYS_23.copy()
scale_name_map = get_mmpi_label_map(keys=scale_keys)


In [ ]:
# 1) 选取 'KEEP' == 1
mmpi = mmpi[mmpi['KEEP'] == 1].copy()

# 2) 保留第12列及之后的所有列 (Python 下标从0开始)
mmpi = mmpi[scale_keys].copy()

# 3) 去除含有 999 的行
mmpi = mmpi[~(mmpi == 999).any(axis=1)].copy()
mmpi

In [ ]:
all_ids = np.sort(mmpi.index.values)
all_ids

In [ ]:
# save_path = "/home/yojiao/hpchome/SFC_project/LNM_voxel/Iowa_Lesion_MMPI.txt"

# with open(save_path, "w") as f:
#     for x in all_ids:
#         f.write(f"{int(x):04d}\n")

# print("Saved:", save_path)

In [ ]:
def plot_corr_mmpi(df):
    import os
    from matplotlib.lines import Line2D

    # 与 plots.py::plot_loadings 保持一致的四类颜色
    category_colors = {
        "Somatic/Cognitive": "#EB7C69",
        "Internalizing": "#6b93c6",
        "Externalizing": "#bd7dc4",
        "Interpersonal": "#4ca866",
    }

    somatic_raw = {
        "Malaise (MLS)", "Gastro-Intestinal Complaints (GIC)", "Head Pain Complaints (HPC)",
        "Neurological Complaints (NUC)", "Cognitive Complaints (COG)"
    }
    internalizing_raw = {
        "Suicidal/Death Ideation (SUI)", "Helplessness/Hopelessness (HLP)", "Self-Doubt (SFD)",
        "Inefficacy (NFC)", "Stress/Worry (STW)", "Anxiety (AXY)", "Anger Proneness (ANP)",
        "Behavior-Restricting Fears (BRF)", "Multiple Specific Fears (MSF)"
    }
    externalizing_raw = {
        "Juvenile Conduct Problems (JCP)", "Substance Abuse (SUB)", "Aggression (AGG)", "Activation (ACT)"
    }
    interpersonal_raw = {
        "Family Problems (FML)", "Interpersonal Passivity (IPP)", "Social Avoidance (SAV)",
        "Shyness (SHY)", "Disaffiliativeness (DSF)"
    }

    somatic = {mmpi_strip_abbr(s) for s in somatic_raw}
    internalizing = {mmpi_strip_abbr(s) for s in internalizing_raw}
    externalizing = {mmpi_strip_abbr(s) for s in externalizing_raw}
    interpersonal = {mmpi_strip_abbr(s) for s in interpersonal_raw}

    def get_category(scale_label):
        normalized = mmpi_strip_abbr(scale_label)
        if normalized in somatic:
            return "Somatic/Cognitive"
        if normalized in internalizing:
            return "Internalizing"
        if normalized in externalizing:
            return "Externalizing"
        if normalized in interpersonal:
            return "Interpersonal"
        return None

    def get_label_color(scale_label):
        cat = get_category(scale_label)
        return category_colors.get(cat, "black")

    # 只取出在 scale_name_map 中、且确实存在于 df 的列
    active_keys = [k for k in scale_name_map.keys() if k in df.columns]
    if not active_keys:
        raise ValueError("当前 scale_name_map 中没有任何列出现在 df 中。")

    # 按字典顺序选列+改名（字典有顺序的话，可以控制展示顺序）
    df_renamed = df[active_keys].rename(columns=scale_name_map)
    scales = df_renamed.columns.tolist()

    fig, ax = plt.subplots(figsize=(16, 14))

    # 计算相关矩阵
    corr_matrix = df_renamed.corr()
    labels = corr_matrix.map(lambda value: f'{0.0 if abs(value) < 0.0005 else value:.3f}')
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), 1)

    sns_heatmap = sns.heatmap(
        corr_matrix, mask=mask, annot=labels, fmt="", cmap='RdBu_r',
        cbar=True, square=True, linewidths=.5, ax=ax,
        cbar_kws={'shrink': 0.5, 'ticks': [-1, 0, 1]},
        vmin=-1, vmax=1, annot_kws={"size": 6.2}
    )
    cbar = sns_heatmap.collections[0].colorbar
    cbar.ax.tick_params(labelsize=14)  # 调大 colorbar 刻度字体

    plt.subplots_adjust(left=0.2, right=0.86, bottom=0.2, top=0.95)

    ax.tick_params(axis='y', length=0)
    ax.set_xticks(np.arange(len(scales)) + 0.5)
    ax.set_xticklabels(scales, rotation=45, ha="right", rotation_mode="anchor", fontsize=14)
    ax.set_yticks(np.arange(len(scales)) + 0.5)
    ax.set_yticklabels(scales, fontsize=14)

    # 根据四类给坐标轴文本上色
    for lbl in ax.get_xticklabels():
        lbl.set_color(get_label_color(lbl.get_text()))
    for lbl in ax.get_yticklabels():
        lbl.set_color(get_label_color(lbl.get_text()))

    # 颜色-类别说明
    handles = [
        Line2D([0], [0], marker='o', color='none', label=cat,
               markerfacecolor=color, markeredgecolor='none', markersize=12)
        for cat, color in category_colors.items()
    ]
    ax.legend(
        handles=handles,
        title='Scale Category',
        loc='best',
        # bbox_to_anchor=(1.0, 1.0),
        frameon=False,
        fontsize=14,
        title_fontsize=16,
    )

    plt.title("Correlations of scores across MMPI-2-RF scales", fontsize=22)

    os.makedirs("figs/supp_mmpi", exist_ok=True)
    fig.savefig("./figs/supp_mmpi/mmpi2rf_corr_scales.pdf", bbox_inches="tight")
    plt.show()

plot_corr_mmpi(mmpi)

In [ ]:
def plot_clustermap_mmpi(df, name_map):
    """
    Hierarchical clustering heatmap of MMPI scales with full names.
    df: DataFrame, index=samples, columns=short names (e.g., tEID, tRCd...)
    name_map: dict mapping short name -> full descriptive name
    """

    # ---- 1. 只保留 name_map 中存在于 df 的列 ----
    active_keys = [k for k in name_map.keys() if k in df.columns]
    if not active_keys:
        raise ValueError("name_map 中的 key 没有任何出现在 df.columns 中。")

    # ---- 2. 选列 + rename ----
    df_renamed = df[active_keys].rename(columns=name_map)

    # ---- 3. 计算相关矩阵 ----
    corr = df_renamed.corr()

    # ---- 4. 画 clustermap ----
    cg = sns.clustermap(
        corr,
        method="average",
        metric="correlation",
        cmap="RdBu_r",
        center=0,
        annot=True,
        fmt=".2f",
        annot_kws={"size":6},
        linewidths=0.3,
        figsize=(12, 12),
        cbar_kws={"label": "Correlation"}
    )

    # ---- 5. 标签调整 ----
    plt.setp(cg.ax_heatmap.get_xticklabels(), rotation=90, fontsize=7)
    plt.setp(cg.ax_heatmap.get_yticklabels(), rotation=0, fontsize=7)

    plt.show()

plot_clustermap_mmpi(mmpi, scale_name_map)


In [ ]:
plt.figure(figsize=(1.2 * len(mmpi.columns), 6))

# ---- Violin plot（梯形图 / 小提琴图） ----
sns.violinplot(
    data=mmpi,
    inner=None,             # 不绘制内部小提琴细节
    color="#89C2D9",        # 浅蓝色
)

# ---- Boxplot（箱线图）叠加 ----
sns.boxplot(
    data=mmpi,
    width=0.15,              # 箱体窄一点
    color="white",
    showcaps=True,
    boxprops={"zorder": 1},
    whiskerprops={"linewidth": 1.5},
    medianprops={"color": "red", "linewidth": 2},
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Ridgeline: 与 cell8 的 violin+box 对比（同一 mmpi 数据）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

if "mmpi" not in globals():
    raise NameError("未检测到变量 mmpi。请先运行前面的数据加载与清洗单元。")

plot_df = mmpi.copy()
plot_df = plot_df.select_dtypes(include=[np.number])
plot_df = plot_df.dropna(axis=1, how="all")

if plot_df.shape[1] == 0:
    raise ValueError("mmpi 中没有可用于绘图的数值列。")

# 与 plots.py::plot_loadings 保持一致的四类颜色
category_colors = {
    "Somatic/Cognitive": "#EB7C69",
    "Internalizing": "#6b93c6",
    "Externalizing": "#bd7dc4",
    "Interpersonal": "#4ca866",
}

somatic_raw = {
    "Malaise (MLS)", "Gastro-Intestinal Complaints (GIC)", "Head Pain Complaints (HPC)",
    "Neurological Complaints (NUC)", "Cognitive Complaints (COG)"
}
internalizing_raw = {
    "Suicidal/Death Ideation (SUI)", "Helplessness/Hopelessness (HLP)", "Self-Doubt (SFD)",
    "Inefficacy (NFC)", "Stress/Worry (STW)", "Anxiety (AXY)", "Anger Proneness (ANP)",
    "Behavior-Restricting Fears (BRF)", "Multiple Specific Fears (MSF)"
}
externalizing_raw = {
    "Juvenile Conduct Problems (JCP)", "Substance Abuse (SUB)", "Aggression (AGG)", "Activation (ACT)"
}
interpersonal_raw = {
    "Family Problems (FML)", "Interpersonal Passivity (IPP)", "Social Avoidance (SAV)",
    "Shyness (SHY)", "Disaffiliativeness (DSF)"
}

somatic = {mmpi_strip_abbr(s) for s in somatic_raw}
internalizing = {mmpi_strip_abbr(s) for s in internalizing_raw}
externalizing = {mmpi_strip_abbr(s) for s in externalizing_raw}
interpersonal = {mmpi_strip_abbr(s) for s in interpersonal_raw}

def get_category(scale_label):
    normalized = mmpi_strip_abbr(scale_label)
    if normalized in somatic:
        return "Somatic/Cognitive"
    if normalized in internalizing:
        return "Internalizing"
    if normalized in externalizing:
        return "Externalizing"
    if normalized in interpersonal:
        return "Interpersonal"
    return None

def get_label_color(scale_label):
    cat = get_category(scale_label)
    return category_colors.get(cat, "black")

# Use centralized display labels from utils.py (controlled by MMPI_LABEL_STYLE)
name_map = globals().get("scale_name_map", {})
full_name_map = {}
for k in plot_df.columns:
    raw = name_map.get(k, k)
    full_name_map[k] = str(raw).strip() if str(raw).strip() else str(k)

scales = plot_df.columns.tolist()
records = []
for c in scales:
    label = full_name_map[c]
    vals = pd.to_numeric(plot_df[c], errors="coerce").to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    for v in vals:
        records.append((label, float(v)))

ridge_df = pd.DataFrame(records, columns=["Scale", "Score"])
order = [full_name_map[c] for c in scales[::-1]]

q01, q99 = ridge_df["Score"].quantile([0.01, 0.99])
pad = 0.08 * (q99 - q01 if q99 > q01 else 1.0)
xmin = q01 - pad
xmax = q99 + pad
x_grid = np.linspace(xmin, xmax, 500)

n_scales = len(order)
fig_h = max(7.0, min(0.24 * n_scales + 1.8, 14.5))

sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(figsize=(12.0, fig_h), dpi=300)


# 同类别采用同色系渐变（深到浅），避免每条山峦完全同色
category_scale_counts = {}
for scale_label in order:
    cat = get_category(scale_label) or "Other"
    category_scale_counts[cat] = category_scale_counts.get(cat, 0) + 1

category_palettes = {}
for cat, n in category_scale_counts.items():
    base_color = category_colors.get(cat, "#9E9E9E")
    if n <= 1:
        category_palettes[cat] = [base_color]
    else:
        # 去掉最亮和最暗端点，保持观感柔和且可区分
        palette = sns.light_palette(base_color, n_colors=n + 2, reverse=True)
        category_palettes[cat] = palette[1:-1][::-1]

category_seen = {cat: 0 for cat in category_scale_counts}
scale_colors = []
for scale_label in order:
    cat = get_category(scale_label) or "Other"
    idx = category_seen[cat]
    scale_colors.append(category_palettes[cat][idx])
    category_seen[cat] += 1
ridge_height = 0.88
ridge_step = 0.62  # 压缩纵向距离，使布局更紧凑
rng = np.random.default_rng(42)

for i, c in enumerate(order):
    vals = ridge_df.loc[ridge_df["Scale"] == c, "Score"].to_numpy()
    if len(vals) < 2:
        continue

    if np.nanstd(vals) < 1e-12:
        vals = vals + rng.normal(0.0, 1e-6, size=len(vals))

    kde = gaussian_kde(vals, bw_method=0.22)
    dens = kde(x_grid)
    dens = dens / dens.max() * ridge_height

    y0 = i * ridge_step
    col = scale_colors[i]
    z = n_scales - i  # 让下面的山峦(较小 y)覆盖上面的山峦

    ax.fill_between(x_grid, y0, y0 + dens, color=col, alpha=0.84, linewidth=0, zorder=z)
    ax.plot(x_grid, y0 + dens, color="#1f1f1f", lw=0.9, alpha=0.9, zorder=z + 0.1)

    med = float(np.median(vals))
    dens_at_med = float(np.interp(med, x_grid, dens))
    ax.vlines(med, y0, y0 + dens_at_med, color="white", lw=1.8, alpha=0.95, zorder=z + 0.2)

ax.set_yticks([i * ridge_step + ridge_height * 0.42 for i in range(n_scales)])
ax.set_yticklabels(order, fontsize=12)

# 根据四类给 y 轴文本上色
for lbl in ax.get_yticklabels():
    lbl.set_color(get_label_color(lbl.get_text()))

# 颜色-类别说明
handles = [
    Line2D([0], [0], marker='o', color='none', label=cat,
           markerfacecolor=color, markeredgecolor='none', markersize=8)
    for cat, color in category_colors.items()
]
ax.legend(
    handles=handles,
    title='Scale Category',
    loc='upper left',
    bbox_to_anchor=(1.01, 1.0),
    frameon=False,
    fontsize=11,
    title_fontsize=12,
)

ax.set_xlabel("T Score", fontsize=14)
ax.set_ylabel("")
ax.set_title("Distribution of scores across MMPI-2-RF scales", fontsize=18, pad=8)

ax.set_xlim(xmin, xmax)
ax.tick_params(axis="x", labelsize=12)
ax.grid(axis="x", linestyle="--", linewidth=0.55, alpha=0.35)
ax.grid(axis="y", visible=False)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# 全称标签较长，给左边留出足够空间
plt.subplots_adjust(left=0.42, right=0.86, top=0.95, bottom=0.06)
os.makedirs("figs/supp_mmpi", exist_ok=True)
fig.savefig("./figs/supp_mmpi/mmpi2rf_distribution_ridgeline.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def pca_variance_curves(df, outfile_prefix="mmpi_pca"):
    """
    df: DataFrame of shape (n_samples, n_features), index=id, columns=scales
    outfile_prefix: prefix to save figures
    """
    # ---- 1) Z-score features ----
    X = df.values
    X = StandardScaler().fit_transform(X)

    # ---- 2) Fit PCA with full number of components ----
    n_feats = X.shape[1]
    pca = PCA(n_components=n_feats, svd_solver="full", random_state=0)
    pca.fit(X)

    evr = pca.explained_variance_ratio_               # per-component variance ratio
    cum = np.cumsum(evr)                               # cumulative ratio
    xs  = np.arange(1, n_feats + 1)

    # ---- 3) Print thresholds ----
    def n_for(thresh):
        return int(np.searchsorted(cum, thresh, side="left") + 1)

    print(f"Components for 70%: {n_for(0.70)}")
    print(f"Components for 80%: {n_for(0.80)}")
    print(f"Components for 90%: {n_for(0.90)}")

    # ---- 4) Scree curve (per-component explained variance) ----
    plt.figure(figsize=(6,4))
    plt.plot(xs, evr, marker="o")
    plt.xlabel("Principal component")
    plt.ylabel("Explained variance ratio")
    plt.title("MMPI PCA: per-component explained variance")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    # plt.savefig(f"{outfile_prefix}_scree.pdf", bbox_inches="tight")
    plt.show()

    # ---- 5) Cumulative curve ----
    plt.figure(figsize=(6,4))
    plt.plot(xs, cum, marker="o")
    plt.xlabel("Principal component")
    plt.ylabel("Cumulative explained variance")
    plt.title("MMPI PCA: cumulative explained variance")
    plt.axhline(0.70, linestyle="--", linewidth=1)
    plt.axhline(0.80, linestyle="--", linewidth=1)
    plt.axhline(0.90, linestyle="--", linewidth=1)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    # plt.savefig(f"{outfile_prefix}_cumulative.pdf", bbox_inches="tight")
    plt.show()

# 用法示例（mmpi 的 index 是 ID，列是 42 个量表）：
pca_variance_curves(mmpi, outfile_prefix="mmpi_pca")

In [ ]:
import nibabel as nib
from tqdm import tqdm

# 假设 LESION_DIR 和 all_ids 已经在前面的 cell 中定义好了
LESION_DIR = '/home/yojiao/LSS/boeslab/Data/Lesion/Iowa/Registry/lesionMasks/Chronic/MNI152/2mm'
# all_ids = [46, 102, 3, ...]  # 这里的 id 是整数

id_to_voxel_count = {}

# 使用 tqdm 显示进度
for subj_id in tqdm(all_ids, desc="Computing lesion size"):
    
    # 核心步骤：将整数 id 格式化为 4 位字符串，左边补零
    # 46 -> '0046', 123 -> '0123'
    subj_str = f"{subj_id:04d}"
    
    # 拼接路径 (假设后缀是 .nii.gz，如果是 .nii 可以加个判断)
    file_path = os.path.join(LESION_DIR, f"{subj_str}.nii.gz")
    
    if os.path.exists(file_path):
        # 加载影像
        img = nib.load(file_path)
        data = img.get_fdata()
        
        # 计算大于0的体素数量 (lesion voxel count)
        voxel_count = np.count_nonzero(data > 0)
        
        id_to_voxel_count[subj_id] = voxel_count
    else:
        print(f"Warning: File not found for ID {subj_id} (searched: {file_path})")
        id_to_voxel_count[subj_id] = np.nan  # 或者设为 0

# 打印结果示例
print(f"Processed {len(id_to_voxel_count)} subjects.")

In [ ]:
# # 1. 将字典转换为 DataFrame
# # columns 定义列名，你可以根据需要修改，比如 ['ID', 'Voxel_Count']
# df = pd.DataFrame(list(id_to_voxel_count.items()), columns=['study_id', 'LesionSize'])

# # (可选) 按照 ID 进行排序，这样 CSV 里的数据看起来更整齐
# df = df.sort_values(by='study_id')

# # 2. 保存为 CSV 文件
# # index=False 代表不把 pandas 的索引（0, 1, 2...）写进去
# output_path = './data/Iowa_Lesion_MMPI_LesionSize.csv'
# df.to_csv(output_path, index=False)

# print(f"CSV saved to {output_path}")

# sLNM and fLNM on atlas

In [ ]:
atlas_path = './atlas/Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_MNI152NLin6Asym_2mm.nii.gz'
fLNM_path = './data/Iowa_fLNM'
sLNM_path = './data/Iowa_sLNM'

In [ ]:
# Historical input generation; the current baseline uses direct T-1 FDC via cca401.preferred.
# ------------------------------
# 1. 加载 atlas & ROI 列表
# ------------------------------
atlas_img = nib.load(atlas_path).get_fdata()
roi_labels = np.unique(atlas_img)
roi_labels = roi_labels[roi_labels > 0]
n_roi = len(roi_labels)
print(f"Atlas ROI 数量: {n_roi}")

# ------------------------------
# 2. 准备容器：存储 (sid, feature_vector)
# ------------------------------
valid_subjects = []
sLNM_list = []
fLNM_list = []
missing_subjects = []

# ------------------------------
# 3. 逐 subject 提取 ROI 平均值
# ------------------------------
for sid in all_ids:
    sid_str = f"{int(sid):04d}"  # 46 → "0046"

    # ---- sLNM ----
    nii_path = os.path.join(sLNM_path, sid_str, f"{sid_str}_disconnection_2mm.nii.gz")
    if not os.path.exists(nii_path):
        print(f"[MISSING sLNM] 找不到文件: {sid_str}")
        missing_subjects.append(sid_str)
        continue

    # ---- fLNM ----
    f_path = os.path.join(fLNM_path, f"{sid_str}.npy")
    if not os.path.exists(f_path):
        print(f"[MISSING fLNM] 找不到文件: {sid_str}")
        missing_subjects.append(sid_str)
        continue

    # ---- 加载 voxel map ----
    x = nib.load(nii_path).get_fdata()

    roi_means = []
    for r in roi_labels:
        vals = x[atlas_img == r]
        vals = vals[np.isfinite(vals)]
        roi_means.append(vals.mean() if len(vals) > 0 else np.nan)

    valid_subjects.append(sid_str)
    sLNM_list.append(roi_means)
    fLNM_list.append(np.median(np.load(f_path), 0))

# ------------------------------
# 4. 转成 numpy array
# ------------------------------
sLNM_features = np.array(sLNM_list)
fLNM_features = np.array(fLNM_list)

# ------------------------------
# 5. 输出最终信息
# ------------------------------
print("\n========== 处理结果 ==========")
print("有效 subject 数量:", len(valid_subjects))
print("缺失 subject 数量:", len(missing_subjects))
print("缺失列表:", missing_subjects)
print("最终 sLNM 特征矩阵 shape:", sLNM_features.shape)
print("最终 fLNM 特征矩阵 shape:", fLNM_features.shape)

In [ ]:
## joseph's sLNM results
# sLNM_path = '/home/yojiao/LSS/boeslab/Data/Lesion/Iowa/Registry/derivedData/wmTracking/lead-dbs_v2/HCP_32_v.1.1/MNI152/2mm'

# # ------------------------------
# # 1. 加载 atlas & ROI 列表
# # ------------------------------
# atlas_img = nib.load(atlas_path).get_fdata()
# roi_labels = np.unique(atlas_img)
# roi_labels = roi_labels[roi_labels > 0]
# n_roi = len(roi_labels)
# print(f"Atlas ROI 数量: {n_roi}")

# # ------------------------------
# # 2. 准备容器：存储 (sid, feature_vector)
# # ------------------------------
# valid_subjects = []
# sLNM_list = []
# fLNM_list = []
# missing_subjects = []

# # ------------------------------
# # 3. 逐 subject 提取 ROI 平均值
# # ------------------------------
# for sid in all_ids:
#     sid_str = f"{int(sid):04d}"  # 46 → "0046"

#     # ---- sLNM ----
#     nii_path = os.path.join(sLNM_path, f"{sid_str}.nii.gz")
#     if not os.path.exists(nii_path):
#         print(f"[MISSING sLNM] 找不到文件: {sid_str}")
#         missing_subjects.append(sid_str)
#         continue

#     # ---- fLNM ----
#     f_path = os.path.join(fLNM_path, f"{sid_str}.npy")
#     if not os.path.exists(f_path):
#         print(f"[MISSING fLNM] 找不到文件: {sid_str}")
#         missing_subjects.append(sid_str)
#         continue

#     # ---- 加载 voxel map ----
#     x = nib.load(nii_path).get_fdata()

#     roi_means = []
#     for r in roi_labels:
#         vals = x[atlas_img == r]
#         vals = vals[np.isfinite(vals)]
#         roi_means.append(vals.mean() if len(vals) > 0 else np.nan)

#     valid_subjects.append(sid_str)
#     sLNM_list.append(roi_means)
#     fLNM_list.append(np.median(np.load(f_path), 0))

# # ------------------------------
# # 4. 转成 numpy array
# # ------------------------------
# sLNM_features = np.array(sLNM_list)
# fLNM_features = np.array(fLNM_list)

# # ------------------------------
# # 5. 输出最终信息
# # ------------------------------
# print("\n========== 处理结果 ==========")
# print("有效 subject 数量:", len(valid_subjects))
# print("缺失 subject 数量:", len(missing_subjects))
# print("缺失列表:", missing_subjects)
# print("最终 sLNM 特征矩阵 shape:", sLNM_features.shape)
# print("最终 fLNM 特征矩阵 shape:", fLNM_features.shape)

# SFC calculation

In [ ]:
# # # ---- FC IDs ----
# FC_ROOT = "/home/yojiao/hpchome/SFC_project/GSP_FC_Lesion_Iowa"
# # fc_ids = [os.path.basename(p).replace(".npy", "") for p in glob.glob(f"{FC_ROOT}/*.npy")]

# # # ---- SC IDs ----
# SC_ROOT = "/home/yojiao/hpchome/SFC_project/HCP_SC_Lesion_Iowa"
# # sc_ids = [d for d in os.listdir(SC_ROOT) if os.path.isdir(os.path.join(SC_ROOT, d))]

In [ ]:
# N = 116   # num of ROIs
# subjects = sorted(all_ids)
# N_JOBS = 80

# def compute_one_subject(sid):
#     # Load SC
#     sc_ml = sio.loadmat(
#         os.path.join(SC_ROOT, sid, f"{sid}.Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_MNI152NLin6Asym_2mm.mean_length.pass.connectivity.mat")
#     )["connectivity"]
#     sc_nc = sio.loadmat(
#         os.path.join(SC_ROOT, sid, f"{sid}.Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_MNI152NLin6Asym_2mm.ncount.pass.connectivity.mat")
#     )["connectivity"]

#     # Load FC and compute subject-level median vector (after row-demean per frame)
#     fc_raw = np.load(os.path.join(FC_ROOT, f"{sid}.npy"))
#     # fc_raw = fc_raw - fc_raw.mean(1, keepdims=True)
#     fc_median = np.median(fc_raw, axis=0)

#     # Make FC matrix for SFC node-wise correlations
#     fc = FC_vec2mat(fc_median, n=N)

#     # Node-wise structure–function coupling
#     ml_pr = np.zeros(N, float)
#     ml_sp = np.zeros(N, float)
#     nc_pr = np.zeros(N, float)
#     nc_sp = np.zeros(N, float)

#     for i in range(N):
#         m = np.ones(N, bool)
#         m[i] = False
#         if np.all(sc_ml[i] == 0) or np.all(fc[i] == 0):
#             ml_pr[i] = 0.0
#             ml_sp[i] = 0.0
#         else:
#             ml_pr[i] = pearsonr(sc_ml[i, m], fc[i, m])[0]
#             ml_sp[i] = spearmanr(sc_ml[i, m], fc[i, m])[0]
#         if np.all(sc_nc[i] == 0) or np.all(fc[i] == 0):
#             nc_pr[i] = 0.0
#             nc_sp[i] = 0.0
#         else:
#             nc_pr[i] = pearsonr(sc_nc[i, m], fc[i, m])[0]
#             nc_sp[i] = spearmanr(sc_nc[i, m], fc[i, m])[0]

#     # Also return vectors for saving (no shared-state mutation in parallel)
#     sc_ml_vec = FC_mat2vec(sc_ml)
#     sc_nc_vec = FC_mat2vec(sc_nc)

#     return ml_pr, ml_sp, nc_pr, nc_sp, sc_ml_vec, sc_nc_vec, fc_median


# # Parallel processing
# results = Parallel(n_jobs=N_JOBS, backend="loky")(
#     delayed(compute_one_subject)(sid) for sid in tqdm(subjects, desc="subjects")
# )

# # Unpack parallel returns
# ml_pr_list, ml_sp_list, nc_pr_list, nc_sp_list, sc_ml_vecs, sc_nc_vecs, fc_vecs = zip(*results)

# # Stack results
# sfc_ml_pr = np.vstack(ml_pr_list)
# sfc_ml_sp = np.vstack(ml_sp_list)
# sfc_nc_pr = np.vstack(nc_pr_list)
# sfc_nc_sp = np.vstack(nc_sp_list)

# # Save standalone variables
# sc_ml_all = np.vstack(sc_ml_vecs)   # shape: (N, n_edges)
# sc_nc_all = np.vstack(sc_nc_vecs)   # shape: (N, n_edges)
# fc_all    = np.vstack(fc_vecs)      # shape: (N, n_edges)

In [ ]:
# ============================================================
# # Lesion feature extraction (voxel-level + atlas-level)
# # Deterministic and efficient, for reproducibility
# # ============================================================
# import nibabel as nib
# from tqdm import tqdm

# # ---------------------------
# # User inputs
# # ---------------------------
# atlas_path = '/home/yojiao/sfc_mmpi/atlas/Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_MNI152NLin6Asym_2mm.nii.gz'
# TRAIN_LESION_DIR = '/home/yojiao/LSS/boeslab/Data/Lesion/Iowa/Registry/lesionMasks/Chronic/MNI152/2mm'

# # ---------------------------
# # Load atlas
# # ---------------------------
# atlas_img = nib.load(atlas_path)
# atlas_data = atlas_img.get_fdata().astype(int)
# atlas_shape = atlas_data.shape
# roi_labels = np.unique(atlas_data)
# roi_labels = roi_labels[roi_labels > 0]  # exclude background
# n_roi = len(roi_labels)

# print(f"Atlas loaded: shape={atlas_shape}, n_ROI={n_roi}")

# # ---------------------------
# # Helper functions
# # ---------------------------
# def load_lesion_mask(subj, folder, multi=False):
#     """Load lesion mask(s) for one subject and combine by OR."""
#     if multi:
#         subj_dir = os.path.join(folder, subj)
#         mask_paths = [os.path.join(subj_dir, f) for f in os.listdir(subj_dir)
#                       if f.endswith('.nii') or f.endswith('.nii.gz')]
#     else:
#         mask_paths = [os.path.join(folder, f"{subj}.nii.gz"),
#                       os.path.join(folder, f"{subj}.nii")]
#         mask_paths = [p for p in mask_paths if os.path.exists(p)]
#     if len(mask_paths) == 0:
#         return np.zeros(atlas_shape, dtype=bool)

#     combined = np.zeros(atlas_shape, dtype=bool)
#     for p in mask_paths:
#         data = nib.load(p).get_fdata()
#         if data.shape != atlas_shape:
#             raise ValueError(f"Shape mismatch for {p}: {data.shape} vs atlas {atlas_shape}")
#         combined |= (data > 0)
#     return combined

# def compute_roi_proportion(lesion_bool, atlas_data, roi_labels):
#     """Compute lesion proportion per ROI (0–1)."""
#     props = np.zeros(len(roi_labels), dtype=float)
#     for i, lbl in enumerate(roi_labels):
#         roi_mask = (atlas_data == lbl)
#         roi_vox = np.count_nonzero(roi_mask)
#         if roi_vox == 0:
#             props[i] = np.nan
#         else:
#             props[i] = np.count_nonzero(lesion_bool & roi_mask) / roi_vox
#     return props

# # ---------------------------
# # Load all lesion masks
# # ---------------------------
# print("Loading train lesions...")
# lesion_train = [load_lesion_mask(s, TRAIN_LESION_DIR, multi=False) for s in tqdm(subjects)]

# # Stack into 4D arrays
# lesion_train = np.stack(lesion_train).astype(bool)

# # ---------------------------
# # Derive common lesion mask (training set)
# # ---------------------------
# common_mask = lesion_train.any(axis=0)  # voxels lesioned in ≥1 training subject
# n_common_vox = np.count_nonzero(common_mask)
# print(f"Common lesion voxels (>=1 subj): {n_common_vox:,} / {np.prod(atlas_shape):,}")

# # Flattened voxel-level lesion features
# lesion_train_flat = lesion_train.reshape(lesion_train.shape[0], -1)

# lesion_mask_train = lesion_train_flat[:, common_mask.ravel()]
# # ---------------------------
# # Atlas-level lesion proportion features
# # ---------------------------
# print("Computing atlas-level lesion proportions...")
# lesion_mask_atlas_train = np.vstack([
#     compute_roi_proportion(m, atlas_data, roi_labels) for m in tqdm(lesion_train)
# ])

In [ ]:
# # Save individually
# OUT_DIR = "./data"
# os.makedirs(OUT_DIR, exist_ok=True)

# np.savez(
#     os.path.join(OUT_DIR, "Iowa_Lesion_MMPI.npz"),
#     ids=np.array(subjects, dtype=int),
#     sfc_ml_pr=sfc_ml_pr,
#     sfc_ml_sp=sfc_ml_sp,
#     sfc_nc_pr=sfc_nc_pr,
#     sfc_nc_sp=sfc_nc_sp,
#     sc_ml=sc_ml_all,
#     sc_nc=sc_nc_all,
#     fc=fc_all,
# )